## Wikipedia Retriever

In [2]:
from langchain_community.retrievers import WikipediaRetriever

In [3]:
# Initialize the retriever
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [4]:
# Define the query
query = "Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without explicit programming language instructions."

# get relevant wikipedia documents
docs = retriever.invoke(query)

In [5]:
docs

[Document(metadata={'title': 'Glossary of artificial intelligence', 'summary': 'This glossary of artificial intelligence is a list of definitions of terms and concepts relevant to the study of artificial intelligence (AI), its subdisciplines, and related fields. Related glossaries include Glossary of computer science, Glossary of robotics, Glossary of machine vision, and Glossary of logic.\n\n', 'source': 'https://en.wikipedia.org/wiki/Glossary_of_artificial_intelligence'}, page_content='This glossary of artificial intelligence is a list of definitions of terms and concepts relevant to the study of artificial intelligence (AI), its subdisciplines, and related fields. Related glossaries include Glossary of computer science, Glossary of robotics, Glossary of machine vision, and Glossary of logic.\n\n\n== A ==\n\nA* search\nPronounced "A-star".\nA graph traversal and pathfinding algorithm which is used in many fields of computer science due to its completeness, optimality, and optimal eff

In [6]:
# print retrieved content
for i, doc in enumerate(docs):
    print(f"-----Result {i+1}-----")
    print(f"Content: \n {doc.page_content}")

-----Result 1-----
Content: 
 This glossary of artificial intelligence is a list of definitions of terms and concepts relevant to the study of artificial intelligence (AI), its subdisciplines, and related fields. Related glossaries include Glossary of computer science, Glossary of robotics, Glossary of machine vision, and Glossary of logic.


== A ==

A* search
Pronounced "A-star".
A graph traversal and pathfinding algorithm which is used in many fields of computer science due to its completeness, optimality, and optimal efficiency.

abductive logic programming (ALP)
A high-level knowledge-representation framework that can be used to solve problems declaratively based on abductive reasoning. It extends normal logic programming by allowing some predicates to be incompletely defined, declared as abducible predicates.

abductive reasoning
Also abduction.
A form of logical inference which starts with an observation or set of observations then seeks to find the simplest and most likely expl

## Vector Store Retriever

In [7]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document

e:\Home_Class\Langchain\LandChain_demo\my_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Step 1: Your source documents
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [9]:
# step 2: Initialize embedding model
embedding = HuggingFaceEmbeddings()

vector_stores = Chroma.from_documents(
    documents=documents,
    embedding=embedding,
    collection_name='my_collection'
)

In [10]:
# step 4: Convert vector stores into retriever
retriever = vector_stores.as_retriever(search_kwargs={"k": 2})

In [11]:
query = "What is chroma use for ?"
results = retriever.invoke(query)


In [12]:
for i, doc in enumerate(results):
    print(f"----Results {i+1}----")
    print(f"Content: \n {doc.page_content}")

----Results 1----
Content: 
 Chroma is a vector database optimized for LLM-based search.
----Results 2----
Content: 
 Embeddings convert text into high-dimensional vectors.


## Maximum Marginal Relevance(MMR)

In [13]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [14]:
from langchain_community.vectorstores import FAISS

In [15]:
# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings()

vector_stores = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [16]:
# Enable MMR in the retriever 
retriever = vector_stores.as_retriever(
    search_type='mmr', # this enables MMR
    search_kwargs={'k': 3, 'lambda_mult': 1} # k = top results, lambda_mult = relevance diversity balance
)

In [17]:
query = "what is langchain ?"
results = retriever.invoke(query)

In [18]:
for i, doc in enumerate(results):
    print(f"----Result {i+1}----")
    print(f"Content: \n {doc.page_content}")

----Result 1----
Content: 
 LangChain is used to build LLM based applications.
----Result 2----
Content: 
 LangChain supports Chroma, FAISS, Pinecone, and more.
----Result 3----
Content: 
 LangChain makes it easy to work with LLMs.


## Multiquery Retriver

In [19]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever

In [20]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [21]:
# Initialize embeddings
embedding_model = HuggingFaceEmbeddings()

# create FAISS vector store
vector_stores = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [22]:
# create retrievers
similarity_retrievers = vector_stores.as_retriever(search_type='similarity', search_kwargs={'k': 5})

In [23]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os

load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=(
        os.getenv("HUGGINGFACEHUB_ACCESS_TOKEN")
    )
)

model = ChatHuggingFace(llm = llm)

In [24]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_stores.as_retriever(search_kwargs={'k':5}),
    llm = model
)

In [25]:
# Query
query = "How to improve energy levels and maintain balance ?"

In [26]:
# Retrieve Results
similarity_results = similarity_retrievers.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [27]:
for i, doc in enumerate(similarity_results):
    print(f"----Results {i+1}----")
    print(f"Content: \n {doc.page_content} \n")


----Results 1----
Content: 
 Drinking sufficient water throughout the day helps maintain metabolism and energy. 

----Results 2----
Content: 
 Regular walking boosts heart health and can reduce symptoms of depression. 

----Results 3----
Content: 
 Mindfulness and controlled breathing lower cortisol and improve mental clarity. 

----Results 4----
Content: 
 Consuming leafy greens and fruits helps detox the body and improve longevity. 

----Results 5----
Content: 
 The solar energy system in modern homes helps balance electricity demand. 



In [28]:
for i, doc in enumerate(multiquery_results):
    print(f"----Results {i+1}----")
    print(f"Content: \n {doc.page_content} \n")

----Results 1----
Content: 
 Drinking sufficient water throughout the day helps maintain metabolism and energy. 

----Results 2----
Content: 
 Consuming leafy greens and fruits helps detox the body and improve longevity. 

----Results 3----
Content: 
 Regular walking boosts heart health and can reduce symptoms of depression. 

----Results 4----
Content: 
 Mindfulness and controlled breathing lower cortisol and improve mental clarity. 

----Results 5----
Content: 
 The solar energy system in modern homes helps balance electricity demand. 

----Results 6----
Content: 
 Photosynthesis enables plants to produce energy by converting sunlight. 



## Contextual Compression Retriever

In [29]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [30]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [31]:
# create a FAISS vector store from the documents
embedding_model = HuggingFaceEmbeddings()
vector_stores = FAISS.from_documents(documents=docs, embedding=embedding_model)

In [32]:
base_retriever = vector_stores.as_retriever(search_kwargs={"k": 5})

In [33]:
# set up the compressor using an LLM
llm = model
compressor = LLMChainExtractor.from_llm(llm)

In [34]:
# create a contextual compression retriver
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [35]:
# Query the retriever
query = "What is photosynthesis ?"
compressed_results = compression_retriever.invoke(query)

In [36]:
for i, doc in enumerate(compressed_results):
    print(f"----Result {i+1}----")
    print(doc.page_content)

----Result 1----
Photosynthesis is the process by which green plants convert sunlight into energy.
----Result 2----
Photosynthesis does not occur in animal cells. (from the context)
----Result 3----
The chlorophyll in plant cells captures sunlight during photosynthesis.
----Result 4----
None of the context is relevant to answer the question about photosynthesis.
